# Phase 4: Scientific Experiments & Statistical Proofs
**Stanford Code In Place — Validation & Integrity Suite**

### Overview & Theory:
An AI model's value in physics depends on its reliability in unseen scenarios. This notebook conducts three critical scientific evaluations using the pre-trained weights from Phase 2 and Phase 3:
1. **Noise Robustness Sweep:** Adding stochastic perturbation ($\sigma \in [0.0, 1.5]$) to check if the model collapses or holds its trajectory.
2. **Long-Horizon Autoregressive Rollout:** Running the model recursively for 100+ steps to observe numerical stability vs. baseline models.
3. **Structural Ablation Matrix:** Systematically removing the Dissipation Layer or Gamma Coupling to isolate and prove their exact utility.

Metrics are saved in raw JSON and CSV formats inside the `results/` subdirectory.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch

# Ensure analysis outputs directories exist
analysis_dirs = ["results/noise_robustness", "results/benchmarks", "results/ablation"]
for d in analysis_dirs:
    os.makedirs(d, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Phase 4 Diagnostics initialized. Loading evaluation matrices on: {device}")

evals code

In [19]:
%%writefile evaluation/stability_analysis.py
"""
LNO-IonTransport Production Pipeline: Long-Horizon Stability Analyzer
"""

import torch
import numpy as np
import matplotlib.pyplot as plt

class StabilityAnalyzer:
    def __init__(self, model, device="cuda"):
        self.model = model.to(device)
        self.device = device

    @torch.no_grad()
    def rollout(self, initial_state, phi, flux, noise, dissipation, gamma, steps=100):
        """
        Executes long-horizon autoregressive forecasting using the structured
        6-parameter mapping signature required by the Yuánlǐ LNO core layers.
        """
        self.model.eval()
        states = []
        current = initial_state.clone().to(self.device)

        # Static background fields or boundary conditions passed contextually
        phi_dev = phi.to(self.device)
        flux_dev = flux.to(self.device)
        noise_dev = noise.to(self.device)
        diss_dev = dissipation.to(self.device)
        gamma_dev = gamma.to(self.device)

        for _ in range(steps):
            outputs = self.model(
                state=current,
                phi=phi_dev,
                flux=flux_dev,
                noise=noise_dev,
                dissipation=diss_dev,
                gamma=gamma_dev
            )

            # Extracting via signature dictionary format
            pred = outputs["next_state"] if isinstance(outputs, dict) else outputs
            states.append(pred.detach().cpu())
            current = pred

        return torch.stack(states)

    def compute_energy(self, states):
        return torch.mean(states ** 2, dim=(-1, -2)).numpy()

    def compute_spectral_radius(self, states):
        radii = []
        for s in states:
            fft = torch.fft.rfft(s, dim=-1)
            radii.append(torch.max(torch.abs(fft)).item())
        return np.array(radii)

    def detect_explosion(self, energy, threshold=100.0):
        return bool(np.max(energy) > threshold)

    def plot_energy_evolution(self, energy):
        """Minimalist high-end cinematic visualization layout."""
        plt.figure(figsize=(7, 4))
        plt.plot(energy, color='#1A1A1A', linewidth=1.5, label='LNO Field Evolution')
        plt.xlabel("Autoregressive Horizon Steps")
        plt.ylabel("System Energy Invariant")
        plt.title("Long-Horizon Dynamic Operator Stability Bounds", fontsize=10, pad=12)
        plt.grid(True, linestyle="--", alpha=0.3)
        plt.legend(frameon=False)
        plt.tight_layout()
        plt.show()

    def analyze(self, initial_state, phi, flux, noise, dissipation, gamma, steps=100):
        states = self.rollout(initial_state, phi, flux, noise, dissipation, gamma, steps)
        energy = self.compute_energy(states)
        spectral_radius = self.compute_spectral_radius(states)
        explosion = self.detect_explosion(energy)

        return {
            "energy": energy,
            "spectral_radius": spectral_radius,
            "explosion_detected": explosion
        }

Overwriting evaluation/stability_analysis.py


In [20]:
%%writefile evaluation/spectral_analysis.py
"""

LNO-IonTransport Production Pipeline: Spectral Operator Energy Field Profiler
"""

import torch
import numpy as np
import matplotlib.pyplot as plt

class SpectralAnalyzer:
    def __init__(self):
        pass

    def fft_spectrum(self, field: torch.Tensor) -> torch.Tensor:
        fft = torch.fft.rfft(field, dim=-1)
        return torch.abs(fft)

    def spectral_entropy(self, field: torch.Tensor, eps: float = 1e-8) -> float:
        spectrum = self.fft_spectrum(field)
        p = spectrum / (torch.sum(spectrum, dim=-1, keepdim=True) + eps)
        entropy = -torch.sum(p * torch.log(p + eps), dim=-1)
        return torch.mean(entropy).item()

    def dominant_modes(self, field: torch.Tensor, topk: int = 5):
        spectrum = self.fft_spectrum(field)
        avg_spectrum = torch.mean(spectrum, dim=0)
        if avg_spectrum.ndim > 1:
            avg_spectrum = torch.mean(avg_spectrum, dim=0)
        vals, idx = torch.topk(avg_spectrum, min(topk, avg_spectrum.shape[0]))
        return idx.cpu().numpy(), vals.cpu().numpy()

    def spectral_radius(self, field: torch.Tensor) -> float:
        return torch.max(self.fft_spectrum(field)).item()

    def compare_spectra(self, field_a: torch.Tensor, field_b: torch.Tensor):
        spec_a = torch.mean(self.fft_spectrum(field_a), dim=0)
        spec_b = torch.mean(self.fft_spectrum(field_b), dim=0)
        return spec_a.detach().cpu(), spec_b.detach().cpu()

    def plot_spectra(self, spec_a, spec_b, label_a="Baseline FNO", label_b="Yuánlǐ LNO"):
        plt.figure(figsize=(8, 4))
        plt.plot(spec_a.numpy(), label=label_a, color='#7F7F7F', alpha=0.6, linestyle='--')
        plt.plot(spec_b.numpy(), label=label_b, color='#B8860B', linewidth=1.5) # Premium gold/amber touch
        plt.xlabel("Frequency Eigenmode Configuration")
        plt.ylabel("Spectral Transform Magnitude")
        plt.title("Operator Spectral Field Preservation Analysis", fontsize=10, pad=12)
        plt.legend(frameon=False)
        plt.grid(True, linestyle="--", alpha=0.3)
        plt.tight_layout()
        plt.show()

    def mode_decay_analysis(self, rollout_states: torch.Tensor) -> np.ndarray:
        decay = []
        for state in rollout_states:
            spectrum = self.fft_spectrum(state)
            high_freq = torch.mean(spectrum[..., -10:])
            decay.append(high_freq.item())
        return np.array(decay)

Overwriting evaluation/spectral_analysis.py


In [22]:
%%writefile evaluation/uq_analysis.py
"""
LNO-IonTransport Production Pipeline: Monte Carlo Dropout Uncertainty Quantification (UQ)
"""

import torch
import numpy as np

class MonteCarloDropoutUQ:
    def __init__(self, model, samples: int = 20, device: str = "cuda"):
        self.model = model
        self.samples = samples
        self.device = device

    def enable_dropout(self):
        """Forces dropout layers to stay active during stochastic inference inference."""
        for module in self.model.modules():
            if module.__class__.__name__.startswith("Dropout"):
                module.train()

    @torch.no_grad()
    def predict(self, state, phi, flux, noise, dissipation, gamma):
        self.model.eval()
        self.enable_dropout()

        predictions = []
        for _ in range(self.samples):
            outputs = self.model(
                state=state.to(self.device),
                phi=phi.to(self.device),
                flux=flux.to(self.device),
                noise=noise.to(self.device),
                dissipation=dissipation.to(self.device),
                gamma=gamma.to(self.device)
            )
            pred = outputs["next_state"] if isinstance(outputs, dict) else outputs
            predictions.append(pred.unsqueeze(0))

        predictions = torch.cat(predictions, dim=0) # Shape: [Samples, Batch, Spatial_Grid]
        mean_prediction = predictions.mean(dim=0)
        epistemic_uncertainty = predictions.std(dim=0)

        return mean_prediction, epistemic_uncertainty

    def uncertainty_score(self, uncertainty: torch.Tensor) -> float:
        return torch.mean(uncertainty).item()

    def instability_confidence_map(self, uncertainty: torch.Tensor) -> np.ndarray:
        """Generates regularized mapping for hidden transport collapse imaging."""
        normalized = uncertainty / (torch.max(uncertainty) + 1e-8)
        return normalized.cpu().numpy()

    def compute_residual_aleatoric(self, mean_pred: torch.Tensor, ground_truth: torch.Tensor) -> torch.Tensor:
        """Maps data-conditioned variance from empirical validation target residuals."""
        return (ground_truth.to(self.device) - mean_pred) ** 2

Overwriting evaluation/uq_analysis.py


# Experiment

In [52]:
import os
import json
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

# Corrected Root & Sub-folder Imports
from dataset_loader import IonTransportDataset
from models.fno_baseline import FNOBaseline
from models.koopman_baseline import KoopmanBaseline
from models.lno_model import LindbladNeuralOperator
from evaluation.metrics import compute_all_metrics

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_benchmarked_models():
    """Initializes and safely loads checkpoints for all operator architectures."""
    print("[*] Initializing model layers for benchmarking...")

    fno = FNOBaseline(in_channels=6, width=64, modes=16).to(DEVICE)
    # KoopmanBaseline has nx=128, in_channels=6, latent_dim=128 as default parameters,
    # so no change is needed for it if those are the intended values.
    koopman = KoopmanBaseline(nx=128, in_channels=6, latent_dim=128).to(DEVICE)
    # Aligning LNO's width to 64 to match the checkpoint saved during training
    lno = LindbladNeuralOperator(in_channels=6, width=64, modes=16).to(DEVICE)

    checkpoints = {
        "FNO": ("results/checkpoints/fno_best.pt", fno),
        "Koopman": ("results/checkpoints/koopman_best.pt", koopman),
        "LNO": ("results/checkpoints/best_lno.pt", lno)
    }

    loaded_models = {}
    for name, (path, model) in checkpoints.items():
        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=DEVICE))
            print(f"    [+] Successfully loaded trained weights for {name}")
        else:
            print(f"    [!] Warning: Checkpoint missing for {name} at '{path}'. Using random initialization fallback.")
        loaded_models[name] = model.eval()

    return loaded_models

@torch.no_grad()
def evaluate_operator(model, loader):
    """Evaluates structural and thermodynamic metrics across the dataset."""
    batch_logs = []

    for batch_tuple in loader:
        inputs_stacked, targets_true = batch_tuple

        # Extract individual components from the stacked input tensor
        state_t = inputs_stacked[:, 0, :].to(DEVICE)
        phi = inputs_stacked[:, 1, :].to(DEVICE)
        flux = inputs_stacked[:, 2, :].to(DEVICE)
        noise = inputs_stacked[:, 3, :].to(DEVICE)
        dissipation = inputs_stacked[:, 4, :].to(DEVICE)
        # Assuming gamma is a scalar per batch element, as per trainer_lno.py
        gamma = inputs_stacked[:, 5, 0].to(DEVICE)

        targets_true = targets_true.to(DEVICE)

        pred = model(state_t, phi, flux, noise, dissipation, gamma)

        # Handle models that return a dictionary (e.g., LNO)
        if isinstance(pred, dict) and "next_state" in pred:
            pred = pred["next_state"]

        metrics = compute_all_metrics(pred, targets_true)
        batch_logs.append(metrics)

    return pd.DataFrame(batch_logs).mean().to_dict()

def main():
    os.makedirs("results/benchmark", exist_ok=True)
    print("\n" + "="*70 + "\n[+] LAUNCHING CORE OPERATOR BENCHMARK PIPELINE\n" + "="*70)

    # Pointed to dataset/val as per directory tree
    try:
        # Corrected keyword argument from 'root' to 'root_dir'
        dataset = IonTransportDataset(root_dir="dataset/val")
        loader = DataLoader(dataset, batch_size=32, shuffle=False, drop_last=False)
        print("[+] Mounted verification data stream from 'dataset/val'")
    except Exception as e:
        print(f"[-] Data Engine Error: {e}. Generating mock evaluation loader for safety.")
        # Create a mock dataset that yields (inputs_stacked, targets_true) tuples
        class MockDataset(torch.utils.data.Dataset):
            def __len__(self):
                return 10 # small number of mock samples
            def __getitem__(self, idx):
                # inputs_stacked: [6, 128]
                inputs_stacked = torch.randn(6, 128)
                # targets_true: [128]
                targets_true = torch.randn(128)
                return inputs_stacked, targets_true

        mock_dataset = MockDataset()
        loader = DataLoader(mock_dataset, batch_size=32, shuffle=False, drop_last=False)

    models = load_benchmarked_models()
    benchmark_results = {}

    for name, model in models.items():
        print(f"\n[*] Running Evaluation Loop -> {name}")
        metrics = evaluate_operator(model, loader)
        benchmark_results[name] = metrics

        for metric_name, value in metrics.items():
            print(f"    -> {metric_name:<25}: {value:.6f}")

    df = pd.DataFrame(benchmark_results).T
    df.to_csv("results/benchmark/fno_vs_lno.csv")

    with open("results/benchmark/fno_vs_lno.json", "w") as f:
        json.dump(benchmark_results, f, indent=4)

    print("\n" + "="*70 + "\n[++++] BENCHMARK COMPLETION SUCCESSFUL\n" + "="*70)

if __name__ == "__main__":
    main()


[+] LAUNCHING CORE OPERATOR BENCHMARK PIPELINE
[+] Mounted verification data stream from 'dataset/val'
[*] Initializing model layers for benchmarking...
    [!] Warning: Checkpoint missing for FNO at 'results/checkpoints/fno_best.pt'. Using random initialization fallback.
    [!] Warning: Checkpoint missing for Koopman at 'results/checkpoints/koopman_best.pt'. Using random initialization fallback.
    [+] Successfully loaded trained weights for LNO

[*] Running Evaluation Loop -> FNO
    -> rmse                     : 0.573130
    -> relative_l2              : 0.950903
    -> mae                      : 0.520822
    -> mass_error               : 66.665235
    -> entropy_error            : 24.671314
    -> instability_index        : 0.000989
    -> dissipation_rate         : 0.000103
    -> spectral_energy          : 0.220555

[*] Running Evaluation Loop -> Koopman
    -> rmse                     : 0.604441
    -> relative_l2              : 1.012528
    -> mae                      : 0.55

In [54]:
r"""
================================================================================
STOCHASTIC THERMAL PERTURBATION ANALYSIS
File: experiments/exp_noise_robustness.py
Description: Evaluates LNO structural stability under out-of-distribution thermal fluctuations.
================================================================================
"""

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from dataset_loader import IonTransportDataset
from models.lno_model import LindbladNeuralOperator

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def run_noise_sweep(model, loader, noise_levels):
    accumulated_errors = []

    # Get a fixed batch from verification stream to apply controlled noise
    for batch_tuple in loader:
        inputs_stacked, targets_true = batch_tuple
        inputs_stacked, targets_true = inputs_stacked.to(DEVICE), targets_true.to(DEVICE)
        break

    print("[*] Launching Perturbation Loop across Noise Spectrum...")
    for idx, sigma in enumerate(noise_levels):
        # Inject Gaussian Noise into the initial state channel (Channel 0)
        perturbed_inputs = inputs_stacked.clone()
        perturbed_inputs[:, 0, :] += torch.randn_like(perturbed_inputs[:, 0, :]) * sigma

        # Exact Gemini Unpacking Engine
        state_t = perturbed_inputs[:, 0, :]
        phi = perturbed_inputs[:, 1, :]
        flux = perturbed_inputs[:, 2, :]
        noise = perturbed_inputs[:, 3, :]
        dissipation = perturbed_inputs[:, 4, :]
        gamma = perturbed_inputs[:, 5, 0]

        pred = model(state_t, phi, flux, noise, dissipation, gamma)
        if isinstance(pred, dict) and "next_state" in pred:
            pred = pred["next_state"]

        # Compute Root Mean Squared Error (RMSE)
        rmse = torch.sqrt(torch.mean((pred - targets_true) ** 2)).item()
        accumulated_errors.append(rmse)
        print(f"    -> Sigma [{sigma:.3f}] | Bound Error (RMSE): {rmse:.6f}")

    return accumulated_errors

def main():
    os.makedirs("results/noise_robustness", exist_ok=True)
    print("\n" + "="*70 + "\n[+] INITIATING STOCHASTIC THERMAL ROBUSTNESS SWEEP\n" + "="*70)

    # Load Model with exact parameters matched by Gemini
    model = LindbladNeuralOperator(in_channels=6, width=64, modes=16).to(DEVICE)
    checkpoint_path = "results/checkpoints/best_lno.pt"

    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
        print("[+] Successfully loaded trained weights for LNO Profile.")
    else:
        print("[!] Warning: Checkpoint missing. Running fallback.")
    model.eval()

    # Mount verification data stream
    dataset = IonTransportDataset(root_dir="dataset/val")
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    noise_levels = np.linspace(0.0, 1.5, 20)
    errors = run_noise_sweep(model, loader, noise_levels)

    np.save("results/noise_robustness/errors.npy", np.array(errors))

    # Leica-Inspired Minimalist Visual Plot
    plt.figure(figsize=(7, 4.5), dpi=300)
    plt.plot(noise_levels, errors, color='#111111', linewidth=2, marker='o', markersize=3, label='LNO Error Bound')
    plt.xlabel("Injected Thermal Fluctuation Strength ($\\sigma$)", fontsize=9, fontweight='medium')
    plt.ylabel("State Vector Forecast RMSE", fontsize=9, fontweight='medium')
    plt.title("OOD Generalization: LNO Stability under Non-Equilibrium Noise", fontsize=10, pad=12, fontweight='bold')
    plt.grid(True, linestyle='--', color='#f0f0f0', alpha=0.8)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig("results/noise_robustness/noise_curve.png", bbox_inches='tight')
    plt.close()
    print("[++++] Noise Evaluation Complete. Visual Diagnostics saved to results/noise_robustness/")

if __name__ == "__main__":
    main()


[+] INITIATING STOCHASTIC THERMAL ROBUSTNESS SWEEP
[+] Successfully loaded trained weights for LNO Profile.
[*] Launching Perturbation Loop across Noise Spectrum...
    -> Sigma [0.000] | Bound Error (RMSE): 0.001300
    -> Sigma [0.079] | Bound Error (RMSE): 0.001875
    -> Sigma [0.158] | Bound Error (RMSE): 0.003068
    -> Sigma [0.237] | Bound Error (RMSE): 0.004487
    -> Sigma [0.316] | Bound Error (RMSE): 0.005852
    -> Sigma [0.395] | Bound Error (RMSE): 0.008039
    -> Sigma [0.474] | Bound Error (RMSE): 0.008696
    -> Sigma [0.553] | Bound Error (RMSE): 0.009140
    -> Sigma [0.632] | Bound Error (RMSE): 0.010097
    -> Sigma [0.711] | Bound Error (RMSE): 0.012860
    -> Sigma [0.789] | Bound Error (RMSE): 0.012853
    -> Sigma [0.868] | Bound Error (RMSE): 0.015565
    -> Sigma [0.947] | Bound Error (RMSE): 0.015467
    -> Sigma [1.026] | Bound Error (RMSE): 0.017345
    -> Sigma [1.105] | Bound Error (RMSE): 0.019419
    -> Sigma [1.184] | Bound Error (RMSE): 0.019326
  

In [55]:
"""
================================================================================
PHYSICAL STRUCTURE ABLATION MATRIX
File: experiments/exp_dissipation_ablation.py
Description: Evaluates the specific mathematical impact of individual channels.
================================================================================
"""

import os
import torch
import pandas as pd
from torch.utils.data import DataLoader
from dataset_loader import IonTransportDataset
from models.lno_model import LindbladNeuralOperator
from evaluation.metrics import compute_all_metrics

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def evaluate_ablated_lno(model, loader, mode="none"):
    batch_logs = []
    for batch_tuple in loader:
        inputs_stacked, targets_true = batch_tuple

        state_t = inputs_stacked[:, 0, :].to(DEVICE)
        phi = inputs_stacked[:, 1, :].to(DEVICE)
        flux = inputs_stacked[:, 2, :].to(DEVICE)
        noise = inputs_stacked[:, 3, :].to(DEVICE)

        # Ablation Strategy Selection
        if mode == "ablate_dissipation":
            dissipation = torch.zeros_like(inputs_stacked[:, 4, :]).to(DEVICE) # Kill dissipation profile
        else:
            dissipation = inputs_stacked[:, 4, :].to(DEVICE)

        if mode == "ablate_gamma":
            gamma = torch.zeros_like(inputs_stacked[:, 5, 0]).to(DEVICE) # Kill coupling scale
        else:
            gamma = inputs_stacked[:, 5, 0].to(DEVICE)

        targets_true = targets_true.to(DEVICE)
        pred = model(state_t, phi, flux, noise, dissipation, gamma)

        if isinstance(pred, dict) and "next_state" in pred:
            pred = pred["next_state"]

        metrics = compute_all_metrics(pred, targets_true)
        batch_logs.append(metrics)

    return pd.DataFrame(batch_logs).mean().to_dict()

def main():
    os.makedirs("results/ablation", exist_ok=True)
    print("\n" + "="*70 + "\n[+] RUNNING CRITICAL LAYER ABLATION MATRIX\n" + "="*70)

    dataset = IonTransportDataset(root_dir="dataset/val")
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    model = LindbladNeuralOperator(in_channels=6, width=64, modes=16).to(DEVICE)
    checkpoint = "results/checkpoints/best_lno.pt"
    if os.path.exists(checkpoint):
        model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    model.eval()

    ablation_results = {}

    print("[*] Evaluating Full LNO Framework Architecture...")
    ablation_results["Full_LNO_Framework"] = evaluate_ablated_lno(model, loader, mode="none")

    print("[*] Evaluating Ablated Dissipation Space Module...")
    ablation_results["Ablated_Dissipation"] = evaluate_ablated_lno(model, loader, mode="ablate_dissipation")

    print("[*] Evaluating Ablated Gamma Coupling Coefficients...")
    ablation_results["Ablated_Gamma_Coupling"] = evaluate_ablated_lno(model, loader, mode="ablate_gamma")

    df = pd.DataFrame(ablation_results).T
    print("\n" + "="*20 + " ABLATION MATRIX RESULTS " + "="*20)
    print(df[['rmse', 'mae', 'entropy_error']])
    print("="*65 + "\n")

    df.to_csv("results/ablation/dissipation_ablation.csv")
    print("[++++] Ablation Matrix committed to disk successfully.")

if __name__ == "__main__":
    main()


[+] RUNNING CRITICAL LAYER ABLATION MATRIX
[*] Evaluating Full LNO Framework Architecture...
[*] Evaluating Ablated Dissipation Space Module...
[*] Evaluating Ablated Gamma Coupling Coefficients...

==================== ABLATION MATRIX RESULTS ====================
                            rmse       mae  entropy_error
Full_LNO_Framework      0.001233  0.000539       0.031984
Ablated_Dissipation     0.063966  0.024297       0.769123
Ablated_Gamma_Coupling  0.002259  0.001735       0.075911

[++++] Ablation Matrix committed to disk successfully.


In [56]:
"""
================================================================================
AUTOREGRESSIVE LONG-HORIZON DRIFT TRACKING
File: experiments/exp_long_horizon.py
Description: Tracks systemic error accumulation across recursive multi-step unrolling.
================================================================================
"""

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from dataset_loader import IonTransportDataset
from models.fno_baseline import FNOBaseline
from models.lno_model import LindbladNeuralOperator

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def run_autoregressive_rollout(model, loader, steps=50):
    for batch_tuple in loader:
        inputs_stacked, _ = batch_tuple
        inputs_stacked = inputs_stacked.to(DEVICE)
        break # Extract a clean verification batch anchor

    # Extract static environment states
    phi = inputs_stacked[:, 1, :]
    flux = inputs_stacked[:, 2, :]
    noise = inputs_stacked[:, 3, :]
    dissipation = inputs_stacked[:, 4, :]
    gamma = inputs_stacked[:, 5, 0]

    # Initial state tracking
    current_state = inputs_stacked[:, 0, :].clone()
    energy_history = []

    for step in range(steps):
        pred = model(current_state, phi, flux, noise, dissipation, gamma)
        if isinstance(pred, dict) and "next_state" in pred:
            pred = pred["next_state"]

        # Autoregressively feed back prediction as next input state
        current_state = pred.detach()

        # Track Kinetic Energy Norm to monitor explosion / decay boundaries
        kinetic_energy = torch.mean(current_state ** 2).item()
        energy_history.append(kinetic_energy)

    return energy_history

def main():
    os.makedirs("results/long_horizon", exist_ok=True)
    print("\n" + "="*70 + "\n[+] INITIALIZING AUTOREGRESSIVE LONG-HORIZON STABILITY EXPERIMENT\n" + "="*70)

    dataset = IonTransportDataset(root_dir="dataset/val")
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    # Initializing matched classes
    fno = FNOBaseline(in_channels=6, width=64, modes=16).to(DEVICE)
    lno = LindbladNeuralOperator(in_channels=6, width=64, modes=16).to(DEVICE)

    if os.path.exists("results/checkpoints/best_lno.pt"):
        lno.load_state_dict(torch.load("results/checkpoints/best_lno.pt", map_location=DEVICE))
        print("[+] Loaded pre-trained weights for LNO Target Layers.")

    fno.eval()
    lno.eval()

    print("[*] Simulating FNO Unbounded Horizon...")
    fno_energy = run_autoregressive_rollout(fno, loader, steps=40)

    print("[*] Simulating LNO Dissipation-Constrained Horizon...")
    lno_energy = run_autoregressive_rollout(lno, loader, steps=40)

    # Leica Design Language Plot
    plt.figure(figsize=(7.5, 4), dpi=300)
    plt.plot(fno_energy, label='Standard FNO (Unconstrained Operator)', color='#e74c3c', linewidth=1.5, linestyle='--')
    plt.plot(lno_energy, label='Lindblad Neural Operator (LNO Boundary)', color='#111111', linewidth=2)
    plt.xlabel("Autoregressive Unrolling Horizons (t -> t+N)", fontsize=9)
    plt.ylabel("Systemic State Trajectory Energy Norm", fontsize=9)
    plt.title("Thermodynamic Trajectory Invariance and Open System Stability Bounds", fontsize=10, pad=10, fontweight='bold')
    plt.legend(frameon=True, facecolor='white', edgecolor='none')
    plt.grid(True, linestyle=':', color='#e0e0e0')
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig("results/long_horizon/energy_stability.png", bbox_inches='tight')
    plt.close()

    print("[++++] Stability Rollouts Compiled. Diagnostic plots committed to results/long_horizon/ plots.")

if __name__ == "__main__":
    main()


[+] INITIALIZING AUTOREGRESSIVE LONG-HORIZON STABILITY EXPERIMENT
[+] Loaded pre-trained weights for LNO Target Layers.
[*] Simulating FNO Unbounded Horizon...
[*] Simulating LNO Dissipation-Constrained Horizon...
[++++] Stability Rollouts Compiled. Diagnostic plots committed to results/long_horizon/ plots.


# Visuals

In [ ]:
# spectral_plots 

import os
import numpy as np
import matplotlib.pyplot as plt

class SpectralVisualizer:
    """
    Advanced spectrum diagnostic engine tracking spectral energy, complex boundaries, and long-horizon stabilization parameters.
    """
    def __init__(self, save_dir="results/spectral"):
        self.save_dir = save_dir
        os.makedirs(self.save_dir, exist_ok=True)

    def plot_spectrum(self, signal, filename="spectral_energy.png"):
        fft_vals = np.fft.fft(signal)
        energy = np.abs(fft_vals)
        freqs = np.fft.fftfreq(len(signal))
        half_len = len(freqs) // 2

        fig, ax = plt.subplots(figsize=(6.5, 2.8), dpi=300)
        ax.plot(freqs[:half_len], energy[:half_len], color='#4A148C', linewidth=1.6)

        ax.set_xlabel("Frequency Mode (k)", fontsize=9)
        ax.set_ylabel("Spectral Power Intensity", fontsize=9)
        ax.set_title("Operator Spectral Energy Distribution", fontsize=10, fontweight='semibold', pad=10, loc='left')
        ax.grid(True, linestyle='--', linewidth=0.5, color='#EAEAEA')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, filename), bbox_inches="tight")
        plt.close()

    def plot_eigenvalue_distribution(self, eigenvalues, filename="eigenvalue_distribution.png"):
        fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=300)

        ax.scatter(np.real(eigenvalues), np.imag(eigenvalues), color='#006064', s=15, alpha=0.75, edgecolors='none', label="Operator Spectrum")

        theta = np.linspace(0, 2*np.pi, 300)
        ax.plot(np.cos(theta), np.sin(theta), color='#D32F2F', linestyle="--", linewidth=1.0, label="Unit Stability Circle Bound")

        ax.set_xlabel("Real Axis Re(λ)", fontsize=9)
        ax.set_ylabel("Imaginary Axis Im(λ)", fontsize=9)
        ax.set_title("Spectral Stability Complex Plane Mapping", fontsize=10, fontweight='semibold', pad=10)
        ax.grid(True, linestyle='--', linewidth=0.5, color='#EAEAEA')
        ax.legend(frameon=True, fontsize=8, loc='lower left')
        ax.axis("equal")

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, filename), bbox_inches="tight")
        plt.close()

    def plot_long_horizon_energy(self, fno_energy, lno_energy, filename="long_horizon_stability.png"):
        steps = np.arange(len(fno_energy))
        fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

        ax.plot(steps, fno_energy, color='#757575', linestyle="--", linewidth=1.5, label="Fourier Neural Operator (FNO)")
        ax.plot(steps, lno_energy, color='#1A1A1A', linewidth=2.0, label="Lindblad Neural Operator (LNO)")

        ax.set_xlabel("Autoregressive Rollout Horizon Steps", fontsize=9)
        ax.set_ylabel("Total Conserved System Energy Norm", fontsize=9)
        ax.set_title("Long-Horizon Dissipation & Stability Analysis", fontsize=10, fontweight='semibold', pad=10, loc='left')

        ax.grid(True, linestyle='--', linewidth=0.5, color='#EAEAEA')
        ax.legend(frameon=True, facecolor='#FFFFFF', edgecolor='none', fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, filename), bbox_inches="tight")
        plt.close()


In [ ]:
# plot_fields
import os
import numpy as np
import matplotlib.pyplot as plt

class TransportFieldPlotter:
    """
    Research-grade field visualization engine configured for
    Nature-style editorial layout and high-fidelity text integration.
    """
    def __init__(self, save_dir="results/figures"):
        self.save_dir = save_dir
        os.makedirs(self.save_dir, exist_ok=True)
        self._apply_journal_style()

    def _apply_journal_style(self):
        plt.rcParams.update({
            'font.family': 'sans-serif',
            'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
            'axes.edgecolor': '#111111',
            'axes.linewidth': 0.8,
            'grid.color': '#EAEAEA',
            'grid.linestyle': '--',
            'grid.linewidth': 0.5,
            'figure.dpi': 300,
            'savefig.dpi': 400
        })

    def plot_prediction_vs_truth(self, ground_truth, prediction, title="LNO Field Prediction", filename="lno_field_prediction.png"):
        x = np.arange(len(ground_truth))
        fig, ax = plt.subplots(figsize=(6.5, 3.5))

        ax.plot(x, ground_truth, color='#1A1A1A', linewidth=2.0, label="Ground Truth")
        ax.plot(x, prediction, color='#D32F2F', linestyle="--", linewidth=1.5, label="LNO Prediction")

        ax.set_xlabel("Spatial Coordinate (x)", fontsize=10, labelpad=6)
        ax.set_ylabel("Ion Density (ρ)", fontsize=10, labelpad=6)
        ax.set_title(title, fontsize=11, fontweight='semibold', pad=10, loc='left')

        ax.legend(frameon=True, facecolor='#FFFFFF', edgecolor='none', fontsize=9)
        ax.grid(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, filename), bbox_inches="tight")
        plt.close()

    def plot_absolute_error(self, ground_truth, prediction, filename="absolute_error_map.png"):
        error = np.abs(ground_truth - prediction)
        x = np.arange(len(error))
        fig, ax = plt.subplots(figsize=(6.5, 2.8))

        ax.fill_between(x, error, color='#0F4C81', alpha=0.15)
        ax.plot(x, error, color='#0F4C81', linewidth=1.5, label="Absolute Error")

        ax.set_xlabel("Spatial Coordinate (x)", fontsize=10, labelpad=6)
        ax.set_ylabel("Reconstruction Error", fontsize=10, labelpad=6)
        ax.set_title("Microscopic Absolute Error Distribution", fontsize=11, fontweight='semibold', pad=10, loc='left')

        ax.grid(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, filename), bbox_inches="tight")
        plt.close()

    def plot_flux_field(self, flux, filename="flux_field.png"):
        x = np.arange(len(flux))
        fig, ax = plt.subplots(figsize=(6.5, 2.8))

        ax.plot(x, flux, color='#2E7D32', linewidth=1.8)
        ax.set_xlabel("Spatial Coordinate (x)", fontsize=10, labelpad=6)
        ax.set_ylabel("Flux Vector (J)", fontsize=10, labelpad=6)
        ax.set_title("Electrodiffusive Flux Field Field-State", fontsize=11, fontweight='semibold', pad=10, loc='left')

        ax.grid(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, filename), bbox_inches="tight")
        plt.close()

    def plot_dissipation_profile(self, dissipation, filename="dissipation_profile.png"):
        x = np.arange(len(dissipation))
        fig, ax = plt.subplots(figsize=(6.5, 2.8))

        ax.plot(x, dissipation, color='#E65100', linewidth=1.8)
        ax.set_xlabel("Spatial Coordinate (x)", fontsize=10, labelpad=6)
        ax.set_ylabel("Dissipation Metric", fontsize=10, labelpad=6)
        ax.set_title("Local Dissipation Energy Functional", fontsize=11, fontweight='semibold', pad=10, loc='left')

        ax.grid(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, filename), bbox_inches="tight")
        plt.close()
